##  Alien Dictionary

You are given a list of strings `words` from an alien language, sorted lexicographically according to the rules of this new language. The goal is to determine the order of the characters in the alien alphabet.

- Each word consists of lowercase English letters.
- The alien alphabet may not match the English alphabet.
- If the order is invalid or cannot be determined, return an empty string.

**Return:**
A string representing the characters in the correct order. If there are multiple valid orders, return any of them. If no valid order exists, return an empty string.

### 🧪 Example 1:
Input: words = ["wrt","wrf","er","ett","rftt"]

Output: "wertf"

### 🧪 Example 2:
Input: words = ["z","x"]

Output: "zx"

### 🧪 Example 3:
Input: words = ["z","x","z"]

Output: "" (No valid order)

**Constraints:**
- 1 <= words.length <= 100
- 1 <= words[i].length <= 100
- words[i] consists of only lowercase English letters

In [ ]:
from typing import List, Dict, Set
from collections import defaultdict

class Solution:
    def alienOrder(self, words: List[str]) -> str:
        # 1. Construir o grafo de precedência
        graph = self.build_graph(words)
        if graph is None:
            return ""  # Ordem inválida
        # 2. Ordenação topológica com DFS recursivo + visited
        return self.topological_sort_dfs(graph)

    def build_graph(self, words: List[str]) -> Dict[str, Set[str]]:
        """Extrai relações de precedência entre caracteres."""
        graph = defaultdict(set)
        all_chars = set(''.join(words))
        for c in all_chars:
            graph[c] = set()
        
        for i in range(len(words) - 1):
            w1, w2 = words[i], words[i+1]
            min_len = min(len(w1), len(w2))
            # Caso inválido: prefixo maior vem antes
            if w1[:min_len] == w2[:min_len] and len(w1) > len(w2):
                return None
            # Encontra a primeira diferença de caracteres
            for a, b in zip(w1, w2):
                if a != b:
                    graph[a].add(b)  # Aresta: a vem antes de b
                    break
        return graph

    def topological_sort_dfs(self, graph: Dict[str, Set[str]]) -> str:
        """
        DFS recursivo para ordenação topológica com detecção de ciclos.
        States: 0=unvisited, 1=visiting (em processamento), 2=visited (completo)
        """
        state = {c: 0 for c in graph}
        order = []
        
        def dfs(node: str) -> bool:
            if state[node] == 1:
                return False  # CICLO: reentramos um nó em processamento
            if state[node] == 2:
                return True  # Já processado, skip
            
            state[node] = 1  # Marca como "visitando"
            # Explora todos os vizinhos primeiro (preorder)
            for neighbor in graph[node]:
                if not dfs(neighbor):
                    return False
            
            state[node] = 2  # Marca como "visitado"
            order.append(node)  # Adiciona APÓS explorar vizinhos (postorder)
            return True  # O(V + E)
        
        # Inicia DFS para cada nó não visitado
        for c in graph:
            if state[c] == 0:
                if not dfs(c):
                    return ""  # Ciclo detectado
        
        # Reverso para obter ordem topológica
        return ''.join(reversed(order))  # O(V)

# Exemplos de uso:
if __name__ == "__main__":
    sol = Solution()
    print(sol.alienOrder(["wrt","wrf","er","ett","rftt"]))  # "wertf"
    print(sol.alienOrder(["z","x"]))  # "zx"
    print(sol.alienOrder(["z","x","z"]))  # ""

# 1️⃣ Specification (Especificação de Engenharia)

**Core do Problema:**
Dado um conjunto de palavras ordenadas segundo um alfabeto alienígena desconhecido, descubra a ordem dos caracteres desse alfabeto a partir das restrições implícitas na ordenação.
O desafio é modelar as relações de precedência entre letras e detectar se existe uma ordem válida (ou se há ciclos/impossibilidade).

**Blueprint (Python Type Hints):**
```python
from typing import List, Dict, Set

class Solution:
    def alienOrder(self, words: List[str]) -> str:
        ...
    def build_graph(self, words: List[str]) -> Dict[str, Set[str]]:
        ...
    def topological_sort_dfs(self, graph: Dict[str, Set[str]]) -> str:
        ...
```

**O que significa `Dict[str, Set[str]]`?**

- `Dict` = Dicionário (estrutura chave-valor)
- `str` (primeira) = tipo da **chave** → Uma string (caractere único)
- `Set[str]` = tipo do **valor** → Um conjunto (set) de strings

**Exemplo prático:**
```python
graph = {
    'w': {'e'},           # 'w' aponta para {'e'} (w vem antes de e)
    'e': {'r'},           # 'e' aponta para {'r'} (e vem antes de r)
    'r': {'t'},           # 'r' aponta para {'t'} (r vem antes de t)
    't': {'f'},           # 't' aponta para {'f'} (t vem antes de f)
    'f': set()            # 'f' não aponta para ninguém
}
```

Cada **chave** é um caractere (como 'w'), e cada **valor** é um **conjunto de caracteres** que devem vir depois dele (como {'e'}).

**Edge Cases & Pegadinhas:**
1. Palavras prefixo umas das outras (ex: ["abc", "ab"]): ordem inválida.
2. Ciclo de dependências (ex: a < b, b < c, c < a): impossível determinar ordem.
3. Letras isoladas (que não aparecem em nenhuma restrição): devem aparecer na resposta.
4. Palavras com letras repetidas ou não contíguas.
5. Palavras com apenas um caractere ou todas iguais.

# 2️⃣ Plan (A Heurística e o Desenho Técnico)

**Brute Force (Intuição):**
Comparar todas as letras de todas as palavras para tentar deduzir a ordem, mas isso não escala e não garante detecção de ciclos.

**Gargalo:**
- Comparar todas as letras gera O(N^2 * L) e não resolve dependências indiretas nem ciclos.

**Heurística Otimizada (DFS + Visited):**
- Construir um grafo dirigido (DAG) onde cada aresta u → v indica que u vem antes de v.
- Usar **DFS recursivo** com três estados (unvisited, visiting, visited) para detectar ciclos.
- Adicionar nós à ordem APÓS explorar todos os vizinhos (postorder), depois reverter (equivalente a ordenação topológica).

**Por que DFS recursivo vence aqui?**
- Detecção natural de ciclos: nós em "visiting" que reentramos significam ciclo.
- Código mais intuitivo e didático para reconstruir a ordem.
- Mesma complexidade que BFS (O(V+E)) mas mais fácil de entender visualmente.

**Visual Trace (Exemplo Didático):**

Dado:
words = ["wrt","wrf","er","ett","rftt"]

Grafo resultante (Dict[str, Set[str]]):
```
graph = {
    'w': {'e'},
    'e': {'r'},
    'r': {'t'},
    't': {'f'},
    'f': set()
}
```

DFS com visited:
```
DFS(w): visita w, descem para e
  DFS(e): visita e, descem para r
    DFS(r): visita r, descem para t
      DFS(t): visita t, descem para f
        DFS(f): visita f, sem vizinhos → order = [f]
      Retorna, order = [f, t]
    Retorna, order = [f, t, r]
  Retorna, order = [f, t, r, e]
Retorna, order = [f, t, r, e, w]
REVERSO: [w, e, r, t, f] → "wertf"
```

Estados do nó:
- **0 (unvisited):** ainda não foi visitado
- **1 (visiting):** em processamento (na stack de recursão)
- **2 (visited):** completamente processado

Se reentramos um nó com estado 1 → CICLO DETECTADO!

# 3️⃣ Implementation (DFS Recursivo + Visited)

```python
from typing import List, Dict, Set
from collections import defaultdict

UN_VISITED = 0
CYCLE_DETECT = 1
VISITED  = 2

class Solution:
    def alienOrder(self, words: List[str]) -> str:
        # 1. Construir o grafo de precedência
        graph = self.build_graph(words)
        if graph is None:
            return ""  # Ordem inválida
        # 2. Ordenação topológica com DFS recursivo
        return self.topological_sort_dfs(graph)

    def build_graph(self, words: List[str]) -> Dict[str, Set[str]]:
        graph = defaultdict(set)
        all_chars = set(''.join(words))
        for c in all_chars:
            graph[c] = set()  # O(K), K = total de caracteres
        for i in range(len(words) - 1):
            w1, w2 = words[i], words[i+1]
            min_len = min(len(w1), len(w2))
            if w1[:min_len] == w2[:min_len] and len(w1) > len(w2):
                return None  # Prefixo inválido: w1 > w2 lexicograficamente
            for a, b in zip(w1, w2):
                if a != b:
                    graph[a].add(b)  # O(1) amortizado
                    break
        return graph  # O(N * L)

    def topological_sort_dfs(self, graph: Dict[str, Set[str]]) -> str:
        # Estados: 0 = unvisited, 1 = visiting, 2 = visited
        state = {c: UN_VISITED for c in graph}  # O(V)
        order = []
        
        def dfs(node: str) -> bool:
            if state[node] == CYCLE_DETECT:
                return False  # CICLO DETECTADO: retornando para um nó em processamento
            if state[node] == VISITED:
                return True  # Já processado completamente
            
            state[node] = CYCLE_DETECT  # Marca como "em processamento"
            for neighbor in graph[node]:  # Para cada vizinho
                if not dfs(neighbor):  # Se há ciclo em qualquer ramo
                    return False
            
            state[node] = VISITED  # Marca como "completamente processado"
            order.append(node)  # Adiciona APÓS explorar todos os vizinhos (postorder)
            return True  # O(V + E) no total
        
        for c in graph:
            if state[c] == UN_VISITED:  # Inicia DFS apenas para nós não visitados
                if not dfs(c):  # Se é um grafo invalido com ciclo
                    return ""  # Ordem inválida
        
        return ''.join(order[::-1])  # Inverte para obter a ordem topológica correta
```

# 4️⃣ Complexity (Veredito Final)
- **Time Complexity:** O(N * L + V + E), onde:
  - N * L: construir o grafo (comparar palavras)
  - V + E: DFS visita cada nó e cada aresta uma única vez
- **Space Complexity:** O(V + E) para o grafo + O(V) para a recursão (altura máxima = V em grafos lineares)

---
**Diferença BFS vs DFS recursivo:**
- BFS (Kahn) é iterativo e mais simples para detecção inline de ciclos (indegree).
- DFS recursivo é mais intuitivo para ver a ordem de exploração e natural para detecção de ciclos (visitando → ciclo!).